# Neural Networks for Security Attack Detection (NSL-KDD)

**UE24CS352A Machine Learning - Mini-Project**

This notebook contains the whole project in one place:

| Part | What it does |
|---|---|
| **Setup** | imports, download the NSL-KDD dataset |
| **Data** | load, explore, encode and scale |
| **Part 1** | reproduce the neural network from the CS229 (Stanford, 2021) paper *Using Neural Network for Security Attacks Detection* |
| **Part 2** | extensions: a tuned ReLU MLP vs a Random Forest, on 5-class and binary tasks |
| **Demo** | classify random test records and flag ALERT / ok |

**The task in one line:** given one network connection record (41 features), decide whether it is *normal* or an *attack*, and if it is an attack, which kind.

> Running the whole notebook takes roughly 10-15 minutes on a CPU (most of the time is Part 1: 10 seeds + a 50-epoch curve).

## 1. Setup

Install anything missing (only needed once), then import the libraries.

In [ ]:
# Uncomment if a library is missing
# %pip install numpy pandas scikit-learn matplotlib joblib

In [ ]:
import json
import os
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler

%matplotlib inline

SEED = 42
DATA = "data"       # dataset goes here
OUT = "results"     # figures and metrics go here
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
np.random.seed(SEED)

### Download the dataset

NSL-KDD is a cleaned-up version of the older KDD'99 intrusion dataset (duplicate records removed).
`KDDTrain+` is what we train on, `KDDTest+` is the exam the model has never seen.

In [ ]:
BASE = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
for f in ["KDDTrain+.txt", "KDDTest+.txt"]:
    dest = os.path.join(DATA, f)
    if not os.path.exists(dest):
        print("downloading", f)
        urllib.request.urlretrieve(BASE + f, dest)
print("done")

## 2. Data

### 2.1 Column names and attack families

The data files have no header row, so we supply the 41 feature names + `label` + `difficulty`.

The dataset names ~40 specific attacks. To keep things explainable we group them into **4 families**:

- **DoS** - flood a server until it crashes
- **Probe** - scan for weak spots / open doors
- **R2L** - a remote outsider tries to get in (e.g. guessing passwords)
- **U2R** - someone already inside tries to become admin

So the 5-class task is **Normal / DoS / Probe / R2L / U2R**.

In [ ]:
COLS = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
        "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
        "logged_in", "num_compromised", "root_shell", "su_attempted",
        "num_root", "num_file_creations", "num_shells", "num_access_files",
        "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
        "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
        "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
        "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
        "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
        "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
        "dst_host_serror_rate", "dst_host_srv_serror_rate",
        "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty"]

ATTACK_GROUP = {"normal": "Normal"}
for a in ["back", "land", "neptune", "pod", "smurf", "teardrop", "apache2",
          "udpstorm", "processtable", "worm", "mailbomb"]:
    ATTACK_GROUP[a] = "DoS"
for a in ["satan", "ipsweep", "nmap", "portsweep", "mscan", "saint"]:
    ATTACK_GROUP[a] = "Probe"
for a in ["guess_passwd", "ftp_write", "imap", "phf", "multihop", "warezmaster",
          "warezclient", "spy", "xlock", "xsnoop", "snmpguess", "snmpgetattack",
          "sendmail", "named"]:
    ATTACK_GROUP[a] = "R2L"
for a in ["buffer_overflow", "loadmodule", "rootkit", "perl", "sqlattack",
          "xterm", "ps", "httptunnel"]:
    ATTACK_GROUP[a] = "U2R"
CLASSES = ["Normal", "DoS", "Probe", "R2L", "U2R"]

### 2.2 Load the data

`load()` reads both files, adds a `group` column (the attack family) and drops the unused `difficulty` column.
The `assert` is a safety check that every label got a family.

In [ ]:
def load():
    tr = pd.read_csv(os.path.join(DATA, "KDDTrain+.txt"), names=COLS)
    te = pd.read_csv(os.path.join(DATA, "KDDTest+.txt"), names=COLS)
    for d in (tr, te):
        d["group"] = d["label"].map(ATTACK_GROUP)
        d.drop(columns=["difficulty"], inplace=True)
    assert tr["group"].notna().all() and te["group"].notna().all()
    return tr, te


tr, te = load()
print("train", tr.shape, "| test", te.shape)
tr.head()

### 2.3 Explore: class distribution

The hard part of this dataset is visible right here: **R2L and U2R are rare in training but much more common in test**,
and the test set also contains attack types that never appear in training.

In [ ]:
dist = pd.DataFrame({"train": tr["group"].value_counts(),
                     "test": te["group"].value_counts()}).reindex(CLASSES)
dist.to_csv(os.path.join(OUT, "class_distribution.csv"))
display(dist)

dist.plot(kind="bar", figsize=(6, 3.5), title="Class distribution (train vs test)")
plt.ylabel("records")
plt.tight_layout()
plt.show()

known_attacks = set(tr["label"]) - {"normal"}
unseen = sorted(set(te["label"]) - set(tr["label"]))
print(f"{len(known_attacks)} attack types in train, {len(unseen)} attack types appear ONLY in test:")
print(unseen)

### 2.4 Preprocess: encode and scale

Two problems to solve before a neural network can use this data:

1. **3 columns are text** (`protocol_type`, `service`, `flag`). **One-hot encoding** turns each into yes/no columns, which takes us from 41 features to **122**.
   We encode train and test *together* so both end up with exactly the same columns.
2. **Numbers live on very different scales** (bytes can be in the millions, rates are 0-1). A **scaler** squeezes them to a similar range.
   The scaler learns its numbers from the **train set only** - if it learned from test data too, that would be *data leakage* (seeing the exam paper before the exam).

`MinMaxScaler` squeezes to 0-1, `StandardScaler` centres to mean 0 / std 1. Part 1 uses Standard (as in the paper), Part 2 uses MinMax.

In [ ]:
def preprocess(tr, te, scaler_cls=MinMaxScaler):
    cat = ["protocol_type", "service", "flag"]
    both = pd.concat([tr.assign(_s="tr"), te.assign(_s="te")])
    both = pd.get_dummies(both, columns=cat)  # shared columns -> no train/test mismatch
    tr_x = both[both["_s"] == "tr"].drop(columns=["_s", "label", "group"])
    te_x = both[both["_s"] == "te"].drop(columns=["_s", "label", "group"])
    scaler = scaler_cls().fit(tr_x)  # fit on train only
    return (scaler.transform(tr_x).astype("float32"),
            scaler.transform(te_x).astype("float32"), list(tr_x.columns))


Xtr, Xte, feats = preprocess(tr, te)
print("features after encoding:", len(feats))
print("Xtr:", Xtr.shape, "| Xte:", Xte.shape)

## 3. A reusable "train and grade" function

`evaluate()` takes any model, trains it, predicts on the test set and reports:

- **accuracy** - % guessed correctly
- **macro F1** - a fairer score that treats every class equally (important because U2R is tiny: a lazy model could ignore it and still look accurate)
- a **classification report** per class
- a **confusion matrix** - a grid of *what the model predicted vs what was actually true*, showing exactly where it gets confused

In [ ]:
def evaluate(name, model, Xtr, ytr, Xte, yte, labels, results, tag):
    # integer-encode labels (sklearn's MLP early stopping fails on string labels)
    idx = {l: i for i, l in enumerate(labels)}
    ytr_i = np.array([idx[v] for v in ytr])
    t0 = time.time()
    model.fit(Xtr, ytr_i)
    fit_s = time.time() - t0
    pred = np.array(labels)[model.predict(Xte)]
    acc = accuracy_score(yte, pred)
    f1 = f1_score(yte, pred, average="macro")
    results[f"{name} | {tag}"] = {"accuracy": acc, "macro_f1": f1, "train_seconds": fit_s}
    print(f"[{tag}] {name}: acc={acc:.4f} macroF1={f1:.4f} ({fit_s:.1f}s)")
    print(classification_report(yte, pred, labels=labels, zero_division=0))
    cm = confusion_matrix(yte, pred, labels=labels)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels, rotation=45)
    ax.set_yticks(range(len(labels)), labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{name} ({tag})")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, f"cm_{name.replace(' ', '_')}_{tag}.png"), dpi=150)
    plt.show()
    return model

## 4. Part 1 - Reproducing the paper's network

**The paper's network: 122 inputs -> 500 hidden (sigmoid) -> 24 outputs (softmax)**

| Setting | Meaning |
|---|---|
| `hidden_layer_sizes=(500,)` | one middle layer with 500 neurons |
| `activation="logistic"` | sigmoid - the rule each neuron uses to decide how strongly to fire |
| `solver="sgd"` | plain gradient descent: guess, measure the error, adjust a little |
| `learning_rate_init=0.5` | how big each adjustment is |
| `momentum=0.0` | no extra tricks, matching the paper |
| `batch_size=1000` | learn from 1000 records at a time, then adjust |
| `alpha=1e-4` | L2 regularisation - a penalty for memorising too much |
| `random_state=seed` | the random starting point of the weights |

**The 24 outputs** = `normal` + 22 attacks seen in training + `other`.
The test set has attack types never seen in training, so they are relabelled `other` (the network cannot name them).

**Why 10 seeds?** The paper warns that results vary with random initialisation, so we repeat the whole experiment 10 times and report mean +/- std.

**Why `partial_fit`?** It trains exactly *one epoch* (one full pass over the training data) per call, so we control the number of epochs
(10, as in the paper) and can record accuracy after every epoch for the curve.

Three numbers are reported per run: train accuracy, 24-class test accuracy, and **Normal/Abnormal** accuracy (ignores *which* attack - only checks "attack or not?").

In [ ]:
def paper_reproduction(tr, te, results, n_seeds=10, curve_epochs=50):
    """Reproduce the reference paper's network (CS229 2021):
    122 inputs -> 500 sigmoid hidden -> 24 softmax outputs
    (normal + 22 known attacks + 'other' for attacks unseen in training),
    mini-batch SGD, lr 0.5, batch 1000, 10 epochs, standardised inputs."""
    Xtr, Xte, _ = preprocess(tr, te, StandardScaler)
    known = sorted(set(tr["label"]) - {"normal"})
    assert len(known) == 22, len(known)
    classes = ["normal"] + known + ["other"]  # 24 outputs
    ytr = tr["label"].values
    yte = np.where(te["label"].isin(["normal"] + known), te["label"], "other")
    n_ab_te = yte != "normal"

    def make(seed):
        return MLPClassifier(hidden_layer_sizes=(500,), activation="logistic",
                             solver="sgd", learning_rate_init=0.5, momentum=0.0,
                             nesterovs_momentum=False, batch_size=1000,
                             alpha=1e-4, random_state=seed)

    def scores(m):
        ptr, pte = m.predict(Xtr), m.predict(Xte)
        return (accuracy_score(ytr, ptr), accuracy_score(yte, pte),
                np.mean((pte != "normal") == n_ab_te))

    runs = []
    for seed in range(n_seeds):
        m = make(seed)
        for _ in range(10):
            m.partial_fit(Xtr, ytr, classes=classes)
        a_tr, a_te, a_bin = scores(m)
        runs.append({"seed": seed, "train_acc": a_tr, "test_acc_24class": a_te,
                     "test_acc_normal_vs_abnormal": a_bin})
        print(f"[paper repro] seed {seed}: train={a_tr:.4f} test24={a_te:.4f} normal/abnormal={a_bin:.4f}")
    te24 = np.array([r["test_acc_24class"] for r in runs])
    teb = np.array([r["test_acc_normal_vs_abnormal"] for r in runs])
    results["paper_reproduction"] = {
        "runs": runs,
        "test_acc_24class": {"mean": te24.mean(), "std": te24.std(), "min": te24.min(), "max": te24.max()},
        "test_acc_normal_vs_abnormal": {"mean": teb.mean(), "std": teb.std(), "min": teb.min(), "max": teb.max()}}
    print(f"[paper repro] 24-class test acc {te24.mean():.4f} +/- {te24.std():.4f}; "
          f"normal/abnormal {teb.mean():.4f} +/- {teb.std():.4f}")

    # accuracy-vs-epoch curves (paper Figure 2), seed 0
    m = make(0)
    tr_curve, te_curve = [], []
    for _ in range(curve_epochs):
        m.partial_fit(Xtr, ytr, classes=classes)
        a_tr, a_te, _ = scores(m)
        tr_curve.append(a_tr); te_curve.append(a_te)
    fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
    ax[0].plot(tr_curve, color="red"); ax[0].set_title("(a) Train")
    ax[1].plot(te_curve, color="blue"); ax[1].set_title("(b) Test")
    for a in ax:
        a.set_xlabel("epoch"); a.set_ylabel("accuracy")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "paper_repro_accuracy_vs_epoch.png"), dpi=150)
    plt.show()
    results["paper_reproduction"]["curve_seed0"] = {"train": tr_curve, "test": te_curve}

Run Part 1 (this is the slow cell: 10 seeds + a 50-epoch curve, roughly 10 minutes on a CPU).

In [ ]:
results = {}
paper_reproduction(tr, te, results)

**How to read the result**

- 24-class test accuracy lands around **68.6%** (paper: ~70%) and Normal/Abnormal around **75.2%** (paper: ~80%) - close to the paper.
- Train accuracy is far higher than test accuracy. That big gap is mostly because **17 attack types in the test set were never seen in training**, so the network cannot name them.
- The paper mentions occasional runs above 85%; across our 10 seeds we saw no such outliers.

## 5. Part 2 - Extensions: ReLU MLP vs Random Forest

**Improved neural network (`mk_mlp`)**

- `(128, 64)` - two hidden layers instead of one
- `relu` - a simpler firing rule that usually trains better than sigmoid
- `adam` - a learning method that adapts its own step size
- `early_stopping=True` - holds out 10% of the training data as a mock test and stops by itself when the mock score has not improved for 8 epochs (like stopping revision when your mock marks stop going up)

**Random Forest baseline (`mk_rf`)** - 200 decision trees (each one a flowchart of yes/no questions) that vote on the answer.

We run both models on two tasks:

- **5-class**: Normal / DoS / Probe / R2L / U2R
- **Binary**: Normal vs Attack

In [ ]:
mk_mlp = lambda: MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu",
                               solver="adam", alpha=1e-4, batch_size=256,
                               learning_rate_init=1e-3, max_iter=60,
                               early_stopping=True, validation_fraction=0.1,
                               n_iter_no_change=8, random_state=SEED)
mk_rf = lambda: RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)

# Part 2 uses Min-Max scaled features (computed in section 2.4)
Xtr, Xte, feats = preprocess(tr, te, MinMaxScaler)

### 5.1 Five-class task

In [ ]:
mlp = evaluate("Neural Network (MLP)", mk_mlp(), Xtr, tr["group"], Xte, te["group"], CLASSES, results, "5-class")

In [ ]:
evaluate("Random Forest", mk_rf(), Xtr, tr["group"], Xte, te["group"], CLASSES, results, "5-class");

### 5.2 Binary task (Normal vs Attack)

In [ ]:
# collapse every attack family into one "Attack" label
ybin_tr = np.where(tr["group"] == "Normal", "Normal", "Attack")
ybin_te = np.where(te["group"] == "Normal", "Normal", "Attack")

evaluate("Neural Network (MLP)", mk_mlp(), Xtr, ybin_tr, Xte, ybin_te, ["Normal", "Attack"], results, "binary");

In [ ]:
evaluate("Random Forest", mk_rf(), Xtr, ybin_tr, Xte, ybin_te, ["Normal", "Attack"], results, "binary");

### 5.3 Neural network training curve (5-class model)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(mlp.loss_curve_, label="train loss")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("MLP training loss")
fig.tight_layout()
fig.savefig(os.path.join(OUT, "mlp_loss_curve.png"), dpi=150)
plt.show()

## 6. Results summary

Everything is also saved to `results/metrics.json`.

In [ ]:
with open(os.path.join(OUT, "metrics.json"), "w") as f:
    json.dump(results, f, indent=2)

pr = results["paper_reproduction"]
summary = pd.DataFrame([
    ["Paper network, 24-class acc (10 seeds)", f"{pr['test_acc_24class']['mean']:.1%} +/- {pr['test_acc_24class']['std']:.1%}", "paper ~70%"],
    ["Paper network, Normal/Abnormal acc", f"{pr['test_acc_normal_vs_abnormal']['mean']:.1%} +/- {pr['test_acc_normal_vs_abnormal']['std']:.1%}", "paper ~80%"],
    ["ReLU MLP, 5-class acc", f"{results['Neural Network (MLP) | 5-class']['accuracy']:.1%}", ""],
    ["Random Forest, 5-class acc", f"{results['Random Forest | 5-class']['accuracy']:.1%}", ""],
    ["ReLU MLP, binary acc", f"{results['Neural Network (MLP) | binary']['accuracy']:.1%}", ""],
    ["Random Forest, binary acc", f"{results['Random Forest | binary']['accuracy']:.1%}", ""],
], columns=["Experiment", "Result (KDDTest+)", "Reference"])
summary

**Takeaways**

- Reproducing the paper's network gets close to its reported accuracy; the gap between train and test comes mainly from attack types in the test set that were never seen in training.
- The tuned ReLU MLP scored higher than the Random Forest on both the 5-class and binary tasks.
- Plain accuracy hides weak spots on rare classes (R2L, U2R) - use the confusion matrices and macro F1 above to see them.

## 7. Live demo

Train the paper's network once (about 20 seconds), then classify random test records.
Each row shows the true label vs the prediction, and the verdict: `ALERT` if the network says it is not normal, otherwise `ok`.
A `<-- wrong` marker means the Normal/Abnormal verdict itself was wrong.

In [ ]:
# Train the paper's 122-500-24 network once (10 epochs, seed 0)
Xtr_s, Xte_s, _ = preprocess(tr, te, StandardScaler)
known = sorted(set(tr["label"]) - {"normal"})
classes24 = ["normal"] + known + ["other"]
ytr_lbl = tr["label"].values
yte_lbl = np.where(te["label"].isin(["normal"] + known), te["label"], "other")

demo_model = MLPClassifier(hidden_layer_sizes=(500,), activation="logistic", solver="sgd",
                           learning_rate_init=0.5, momentum=0.0, nesterovs_momentum=False,
                           batch_size=1000, alpha=1e-4, random_state=0)
for _ in range(10):
    demo_model.partial_fit(Xtr_s, ytr_lbl, classes=classes24)

pred = demo_model.predict(Xte_s)
print(f"Full test set: 24-class accuracy {np.mean(pred == yte_lbl):.1%}, "
      f"Normal/Abnormal accuracy {np.mean((pred == 'normal') == (yte_lbl == 'normal')):.1%}")

In [ ]:
def show_samples(n=12, seed=1):
    idx = np.random.default_rng(seed).choice(len(Xte_s), n, replace=False)
    print(f"{'#':>3} {'protocol':8} {'service':10} {'true':16} {'predicted':16} verdict")
    for k, i in enumerate(idx, 1):
        row = te.iloc[i]
        verdict = "ALERT" if pred[i] != "normal" else "ok"
        mark = "" if (pred[i] == "normal") == (yte_lbl[i] == "normal") else "  <-- wrong"
        print(f"{k:>3} {row['protocol_type']:8} {row['service']:10} {yte_lbl[i]:16} {pred[i]:16} {verdict}{mark}")


show_samples(n=12, seed=1)   # change n / seed for other samples